### 질문 저장하기
---

In [4]:
from models import Question, Answer
from datetime import datetime

q = Question(subject='pybo가 무엇인가요?', content='pybo에 대해서 알고 싶습니다.', create_date=datetime.now(), modify_date=datetime.now())

In [ ]:
from database import SessionLocal

# db 객체는 Database와 연결된 세션, 즉 접속된 상태를 의미
db = SessionLocal()

db.add(q)
db.commit()

In [7]:
q = Question(subject='FastAPI 모델 질문입니다.', content='id는 자동으로 생성되나요?', create_date=datetime.now(), update_date=datetime.now())

db.add(q)
db.commit()

q.id

2

### 데이터 조회하기
---

In [2]:
from models import Question, Answer
from database import SessionLocal

db = SessionLocal()
db.query(Question).all()

In [13]:
questions = db.query(Question).filter(Question.id==2).all()
for question in questions:
    print(q.__dict__)


{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x111841cd0>, 'subject': 'pybo가 무엇인가요?', 'content': 'pybo에 대해서 알고 싶습니다.', 'create_date': datetime.datetime(2026, 6, 10, 16, 51, 15, 282910), 'modify_date': datetime.datetime(2026, 6, 10, 16, 51, 15, 282913)}


In [14]:
# SQLAlchemy 2.x 에서 변경된 사항
# db.query(Question).get(1)
db.get(Question, 1)

In [14]:
db.query(Question).filter(Question.subject.like('%FastAPI%')).all()

# ilike: like 사용시 대소문자 구분하지 않고 조회
db.query(Question).filter(Question.subject.ilike('%fastapi%')).all()


### 데이터 수정하기
---

In [ ]:
q = db.get(Question, 2)

q.subject = 'FastAPI Model Question'
q.update_date = datetime.now()

db.commit()

### 데이터 삭제하기
---

In [19]:
q = db.get(Question, 1)
db.delete(q)

db.commit()

In [20]:
db.query(Question).all()

### 답변 데이터 저장하기
---

In [1]:
from datetime import datetime
from models import Question, Answer
from database import SessionLocal

db = SessionLocal()

In [ ]:
q = db.get(Question, 2)

a = Answer(question=q, content='네 자동으로 생성됩니다.', create_date=datetime.now(), update_date=datetime.now())
db.add(a)
db.commit()

In [6]:
a.id
a = db.get(Answer, 1)
a

### 답변에 연결된 질문 찾기 vs 질문에 달린 답변 찾기
---

In [7]:
a.question

In [9]:
q.answers

임시 질문 데이터 300개 생성하기
---

In [3]:
from database import SessionLocal
from models import Question
from datetime import datetime

db = SessionLocal()
for i in range(300):
    q = Question(subject='테스트 데이터입니다:[%03d]' % i, content='내용무', create_date=datetime.now(), update_date=datetime.now())
    db.add(q)
db.commit()

In [4]:
SECRET_KEY = "4ab2fce7a6bd79e1c014396315ed322dd6edb1c5d975c6b74a2904135172c03c"
len(SECRET_KEY)

64

```
$ openssl rand -hex 32
4ab2fce7a6bd79e1c014396315ed322dd6edb1c5d975c6b74a2904135172c03c
```

```
>>> import secrets
>>> secrets.token_hex(32)
'344a451d26d1968c0cd4ca12613e5f61b0f71dafced442c730edba55bb9035bc'
```

In [15]:
from models import User
from database import SessionLocal

# db 객체는 Database와 연결된 세션, 즉 접속된 상태를 의미
db = SessionLocal()

user = db.query(User).filter(User.username == 'lee1').first()
print(user)

### 조인

In [29]:
# query 만 보기
user = db.query(User).filter(User.username=='lee1').first()
query = db.query(Question).filter(Question.user_id==user.id)
print(query)
print('-'*60)


# 결과 보기 - first() 를 쓰고 안쓰고의 차이
user = db.query(User).filter(User.username=='lee1').first()
query = db.query(Question).filter(Question.user_id==user.id).first()
print(query)
print('-'*60)

# 조인 SQL 보기
result = db.query(Question).join(User).filter(User.username == 'lee1')
print(result)
print('-'*60)

# 조인 결과 보기
results = db.query(Question).join(User).filter(User.username == 'lee1').all()
for result in results:
    print({"id": result.id,
           "subject": result.subject,
           "content": result.content
    })
print('-'*60)

SELECT question.id AS question_id, question.subject AS question_subject, question.content AS question_content, question.create_date AS question_create_date, question.modify_date AS question_modify_date, question.user_id AS question_user_id, question.sta AS question_sta 
FROM question 
WHERE question.user_id = ?
------------------------------------------------------------
------------------------------------------------------------
SELECT question.id AS question_id, question.subject AS question_subject, question.content AS question_content, question.create_date AS question_create_date, question.modify_date AS question_modify_date, question.user_id AS question_user_id, question.sta AS question_sta 
FROM question JOIN user ON user.id = question.user_id 
WHERE user.username = ?
------------------------------------------------------------
{'id': 307, 'subject': '수수요일은 뭐해?', 'content': '퇴근하고 수요일은 뭐할까? 로 수정\n'}
------------------------------------------------------------


### 아우터 조인

In [50]:
from database import SessionLocal
db = SessionLocal()

from models import Question, Answer

print(f'Question Count: {db.query(Question).count()}')
print(f'Answer Count: {db.query(Answer).count()}')

# Join
print(f'Question Join Answer Count: {db.query(Question).join(Answer).count()}')

# Outer Join - LEFT OUTER JOIN
print(f'Question OuterJoin Answer Count: {db.query(Question).outerjoin(Answer).count()}')

# distinct() 
print(f'Question OuterJoin Answer Distinct Count: {db.query(Question).outerjoin(Answer).distinct().count()}')

# Outer Join with ilike(대소문자 무시)
query = db.query(Question).outerjoin(Answer).filter(
    Question.content.ilike('%화요일%') | 
    Answer.content.ilike('%화요일%')).distinct().count()
print(f'Question OuterJoin Answer with ilike Count: {query}')


Question Count: 308
Answer Count: 26
Question Join Answer Count: 26
Question OuterJoin Answer Count: 325
Question OuterJoin Answer Distinct Count: 308
Question OuterJoin Answer with ilike Count: 3


In [38]:
query = db.query(Question).outerjoin(Answer).distinct()
print(query)

SELECT DISTINCT question.id AS question_id, question.subject AS question_subject, question.content AS question_content, question.create_date AS question_create_date, question.modify_date AS question_modify_date, question.user_id AS question_user_id, question.sta AS question_sta 
FROM question LEFT OUTER JOIN answer ON question.id = answer.question_id


### 서브쿼리(SubQuery)

In [58]:
# Cartesian Product
sub_query = db.query(Answer.question_id, Answer.content, User.username)
print(sub_query)
print('-' * 60)

sub_query = db.query(Answer.question_id, Answer.content, User.username)\
    .outerjoin(User, Answer.user_id == User.id)
print(sub_query)
print('-' * 60)
    
sub_query = db.query(Answer.question_id, Answer.content, User.username)\
    .outerjoin(User, Answer.user_id == User.id).subquery()
print(sub_query)
print('-' * 60)

# Question 과 sub_query 의 Outer Join
# sub_query.c.question_id 는 서브쿼리 조회 항목 중 quesiont_id를 의미
result = db.query(Question).outerjoin(sub_query, sub_query.c.question_id == Question.id)
print(result)
print('-' * 60)

result = db.query(Question).outerjoin(sub_query, sub_query.c.question_id == Question.id).distinct()
print(result)



SELECT answer.question_id AS answer_question_id, answer.content AS answer_content, user.username AS user_username 
FROM answer, user
------------------------------------------------------------
SELECT answer.question_id AS answer_question_id, answer.content AS answer_content, user.username AS user_username 
FROM answer LEFT OUTER JOIN user ON answer.user_id = user.id
------------------------------------------------------------
SELECT answer.question_id, answer.content, "user".username 
FROM answer LEFT OUTER JOIN "user" ON answer.user_id = "user".id
------------------------------------------------------------
SELECT question.id AS question_id, question.subject AS question_subject, question.content AS question_content, question.create_date AS question_create_date, question.modify_date AS question_modify_date, question.user_id AS question_user_id, question.sta AS question_sta 
FROM question LEFT OUTER JOIN (SELECT answer.question_id AS question_id, answer.content AS content, user.usernam

In [65]:
keyword = '인생'
search = '%%{}%%'.format(keyword)
print(search)
search1 = f'%%{keyword}%%'
print(search1)


%%인생%%
%%인생%%


In [89]:
from models import Question, Answer
from database import SessionLocal

db = SessionLocal()

keyword='화요일'
search = '%%{}%%'.format(keyword)

print(f'Answer Count: {db.query(Answer).count()}')
print(f'User Count: {db.query(User).count()}')

# 단계1
sub_query = db.query(Answer.question_id, Answer.content, User.username).count()
print(sub_query)
print('-'*60)

# 테스트
# ON 절을 안 주어도 FK 관계로 인해 ON 절이 자동으로 들어감(ON user.id = answer.user_id)
sub_query = db.query(Answer.question_id, Answer.content, User.username)\
    .outerjoin(User)
print(f'테스트: {sub_query}')
print('-'*60)

# 단계2    
sub_query = db.query(Answer.question_id, Answer.content, User.username)\
    .outerjoin(User).subquery()
    # .outerjoin(User, Answer.user_id == User.id).subquery()
print(sub_query)
print('-'*60)

# 단계3    
sub_query = db.query(Answer.question_id, Answer.content, User.username)\
    .outerjoin(User).subquery()
question_list = db.query(Question)\
    .outerjoin(User)\
    .outerjoin(sub_query, sub_query.c.question_id == Question.id)
        
print(f'단계3: {question_list}')
print('-'*60)

# 단계4    
sub_query = db.query(Answer.question_id, Answer.content, User.username)\
    .outerjoin(User).subquery()
question_list = db.query(Question)\
    .outerjoin(User)\
    .outerjoin(sub_query, sub_query.c.question_id == Question.id)\
    .filter(Question.subject.ilike(search) |    # 질문제목
            Question.content.ilike(search) |    # 질문내용
            User.username.ilike(search) |       # 질문작성자
            sub_query.c.content.ilike(search) | # 답변내용
            sub_query.c.username.ilike(search)  # 답변작성자
            )\
    .distinct()
        
print(question_list)
print('-'*60)


Answer Count: 26
User Count: 4
104
------------------------------------------------------------
테스트: SELECT answer.question_id AS answer_question_id, answer.content AS answer_content, user.username AS user_username 
FROM answer LEFT OUTER JOIN user ON user.id = answer.user_id
------------------------------------------------------------
SELECT answer.question_id, answer.content, "user".username 
FROM answer LEFT OUTER JOIN "user" ON "user".id = answer.user_id
------------------------------------------------------------
단계3: SELECT question.id AS question_id, question.subject AS question_subject, question.content AS question_content, question.create_date AS question_create_date, question.modify_date AS question_modify_date, question.user_id AS question_user_id, question.sta AS question_sta 
FROM question LEFT OUTER JOIN user ON user.id = question.user_id LEFT OUTER JOIN (SELECT answer.question_id AS question_id, answer.content AS content, user.username AS username 
FROM answer LEFT OUTER

In [ ]:
db.query(Answer.question_id, Answer.content, Answer.)